In [ ]:
# Enable inline plots in the notebook
%matplotlib inline

# Import library functions needed
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from IPython.display import clear_output
#from IPython.core.debugger import set_trace # Activates debugging features

def rasterplot(ax, x, y, x_label, y_label):
# Function used to plot spike times
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.scatter(x, y, marker='|')
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    

# Set default figure size (Might not be needed?)
plt.rcParams['figure.figsize'] = [10, 10]


In [ ]:

"""
 Dataset Generator
    
Generates a sequence of frames showing a single pixel
moving horizontally across the frame.

Parameters:
    frame_size (int): Width and height of square frame.
    seq_length (int): Number of frames in sequence.
    direction (str): "left" or "right".
    speed (int): Pixels moved per frame.

Returns:
    numpy array of shape (seq_length, frame_size, frame_size)
"""

def generate_motion_sequence(
    frame_size = 32,
    seq_length = 20,
    direction = "right",
    speed = 1
):
    frames = []
    y = np.random.randint(5, frame_size-5) 
    
    if direction == "right":
        x = 0
        dx = speed
    else:
        x = frame_size - 1
        dx = -speed
    
    for t in range(seq_length):
        frame = np.zeros((frame_size, frame_size))
        frame[y, int(x)] = 1.0
        frames.append(frame)
        x += dx
    return np.array(frames)


def create_dataset(n_samples = 1000):
    x = []      #Input / motion sequences
    y = []      #Lables
    
    for _ in range(n_samples):
        direction = np.random.choice(["left", "right"])
        speed = np.random.randint(1, 3)
        seq = generate_motion_sequence(direction=direction, speed=speed)
        x.append(seq)
        y.append(0 if direction == "left" else 1)
    return np.array(x), np.array(y)
        

## Delta Modulation (Spike Encoding)
Convert pixel changes to spikes

In [ ]:
def encode_to_spikes(frames, threshold=0.1):
    """
    Convert frame sequence to spikes using delta modulation.
    Spikes generated when pixel change exceeds threshold.
    
    frames: (T, H, W) - sequence of frames
    Returns: (T, H, W) - binary spike array
    """
    spikes = np.zeros_like(frames)
    
    # Generate spikes for temporal differences
    for t in range(1, len(frames)):
        diff = frames[t] - frames[t-1]
        spikes[t] = (np.abs(diff) > threshold).astype(float)
    
    return spikes

In [ ]:
# Example: test with your friend's data generator
# frames = generate_motion_sequence(direction="right", seq_length=30)
# spikes = encode_to_spikes(frames, threshold=0.1)
# print(f"Frame shape: {frames.shape}")
# print(f"Spike shape: {spikes.shape}")
# print(f"Total spikes: {spikes.sum()}")

## Simple SNN (Leaky Integrate-and-Fire)

In [ ]:
def lif_neuron(input_spikes, weights, threshold=1.0, decay=0.9):
    """
    Simple LIF neuron that processes spike input.
    
    input_spikes: (T, N) - spike trains over time
    weights: (N,) - synaptic weights
    Returns: output spikes over time
    """
    T = len(input_spikes)
    membrane = 0.0
    output_spikes = []
    
    for t in range(T):
        # Integrate input
        membrane = membrane * decay + np.dot(input_spikes[t], weights)
        
        # Fire if threshold reached
        if membrane >= threshold:
            output_spikes.append(1)
            membrane = 0  # Reset
        else:
            output_spikes.append(0)
    
    return np.array(output_spikes)

In [ ]:
# Test LIF neuron
# test_spikes = np.random.rand(50, 10) > 0.8  # Random sparse spikes
# test_weights = np.random.randn(10) * 0.5
# output = lif_neuron(test_spikes, test_weights)
# print(f"Output spikes: {output.sum()} out of {len(output)} timesteps")